In [1]:
import os
from dotenv import load_dotenv

from google import genai

load_dotenv()

client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))

response = client.models.generate_content(
    model = "gemini-3.5-flash",
    contents="Explain RAG in 3 words"
)

print(response.candidates[0].content.parts[0].text)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


**Search, then generate.**


In [2]:
response.usage_metadata

GenerateContentResponseUsageMetadata(
  candidates_token_count=6,
  prompt_token_count=8,
  prompt_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=8
    ),
  ],
  thoughts_token_count=670,
  total_token_count=684
)

In [3]:
def llm(prompt):
    response = client.models.generate_content(
        model = "gemini-3.5-flash",
        contents=prompt
    )

    return response.text

In [4]:
llm("Hey,what's up")

"Hey! Not much, just here and ready to help. How's your day going? What can I do for you today?"

In [5]:
question = "I just discovered this course. Can i still join and earn a certificate"
response = llm(question)
print(response)

To give you the most accurate answer, **could you tell me the name of the course or the platform it is hosted on** (e.g., Coursera, edX, Udemy, a specific university website)? 

Since I don't know which specific course you are looking at, here is how it usually works for most online courses:

### 1. If it is a Self-Paced Course (e.g., Udemy, Coursera, edX)
* **Can you join?** **Yes!** You can enroll at any time. There is no start or end date.
* **Can you earn a certificate?** **Yes.** Once you complete all the lessons, quizzes, and assignments, you will automatically receive a certificate.
* *Note:* Some platforms let you learn for free (audit), but you have to pay a fee if you want the official certificate.

### 2. If it is a Session-Based or Cohort-Based Course (e.g., University-led courses)
* **Can you join?** It depends on the deadline. If the enrollment window is still open, you can join. If the course has already started and the deadline has passed, you may have to wait for the n

In [6]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""


question = "Can i earn money from that course"
prompt = f"""
    Your task is to answer the questions from the course participants based on the given context
    Use the context to find relevant information and provide accurate answer to the user
    if there is no answer in the context do not make up things just tell the user that you don't know

    Question: {question}

    Context: {context}
"""

response = llm(prompt)
print(response)

Based on the provided context, there is no information about whether you can earn money from the course. Therefore, I do not know the answer to this question.


In [7]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

In [8]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1400

In [9]:
documents[0]

{'id': '4487db3924',
 'course': 'ai-dev-tools-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'How do I access the course modules and materials?',
 'answer': "All course materials are in the GitHub repo. Each module has its own folder (e.g. `01-overview`, `03-mcp`), and cohort-specific homework is under the `cohorts/` folder.\n\nFor the 2026 cohort, the week-by-week folders are:\n\n- Week 1: [01-ai-native-workflow](https://github.com/DataTalksClub/ai-dev-tools-zoomcamp/tree/main/cohorts/2026/01-ai-native-workflow)\n- Week 2: [02-development](https://github.com/DataTalksClub/ai-dev-tools-zoomcamp/tree/main/cohorts/2026/02-development)\n- Week 3: [03-deployment](https://github.com/DataTalksClub/ai-dev-tools-zoomcamp/tree/main/cohorts/2026/03-deployment)\n- Week 4: [04-devops](https://github.com/DataTalksClub/ai-dev-tools-zoomcamp/tree/main/cohorts/2026/04-devops)\n\nStart from the folder for your current week — it gathers everything for that week in one place.\n\n

In [10]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [11]:
question = "I just discovered the course. Can I join now?"

search_results = index.search(
    question,
    boost_dict={"question": 2.0, "section": 0.5},
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '5cc511f85b',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Does the course certificate show the number of course hours?',
  'answer': 'No. The certificate does not 

In [12]:
[doc["question"] for doc in search_results]

['I just discovered the course. Can I still join?',
 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
 'Does the course certificate show the number of course hours?',
 "Why do we need orchestration / Kestra — can't I just run the code in a notebook?",
 'Certificate: Can I follow the course in a self-paced mode and get a certificate?']

In [13]:
def search(question, course="llm-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

In [14]:
search_results = search(question)
print(search_results)

[{'id': '74eb249bbf', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'I just discovered the course. Can I still join?', 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}, {'id': '977bf7786c', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?', 'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."}, {'id': '5cc511f85b', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Does the course certificate show the number of course hours?', 'answer': 'No. The certificate does not state a total number of ho

In [15]:
from google.genai import types

system_instruction = """
Your task is to answer the questions from the course participants based on the given context
    Use the context to find relevant information and provide accurate answer to the user
    if there is no answer in the context do not make up things just tell the user that you don't know
"""

def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

config = types.GenerateContentConfig(
    temperature =0.2,
    system_instruction=system_instruction
)

def build_prompt(prompt,search_results):
    context = build_context(search_results)

    result = f"""
            Question: {prompt}
            Context: {context}
        """
    return result


In [16]:
prompt = build_prompt(question,search_results)

print(prompt)


            Question: I just discovered the course. Can I join now?
            Context: General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Does the course certificate show the number of course hours?
A: No. The certificate does not state a total number of hours.

Module 3: Orchestration
Q: Why do we need orchestration / Kestra — can't I just run the code in a notebook?
A: Notebooks are great for learning and expe

In [22]:
def llm_with_rag(question):
    search_results = search(question)
    prompt = build_prompt(question,search_results)
    response = client.models.generate_content(
        model = "gemini-3.5-flash",
        contents=prompt,
        config=config
    )
    return response.text

In [23]:
response = llm_with_rag(question)
response

'Yes, you can still join! However, if you want to receive a certificate, you will need to submit your project while submissions are still being accepted.'

In [24]:
response = llm_with_rag("Yo how can I go from my home to school")
response

"I don't know. The provided context does not contain information on how to travel from your home to school."